In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.silver")

#### Nota de negócio: colunas de origem não utilizadas

Ao mapear os CSVs para a Silver, ficaram de fora **`tconst`** (`tb_movies_info`) e **`production_countries`**,
**`spoken_languages`**, **`keywords`** (`tb_credits_and_tags`), nenhuma está no mapeamento definido.

- `tconst` (ID do IMDb) seria útil como chave rápida para cruzar com bases externas do IMDb.
- `keywords` encaixaria muito bem para enriquecer o `llm_context_document` futuro.

Ambas ficam de fora porque o escopo é um requisito de cliente com mapeamento e schema determinísticos.

In [0]:
df_info = spark.table("workspace.bronze.tb_movies_info")

status_map = {
    "RELEASED": "Lançado",
    "POST PRODUCTION": "Pós-Produção",
    "IN PRODUCTION": "Em Produção",
    "PLANNED": "Planejado",
    "RUMORED": "Rumores",
    "CANCELED": "Cancelado",
}

status_map_expr = F.create_map(*[F.lit(x) for item in status_map.items() for x in item])
# Normaliza antes de traduzir e marca valores não informados ou não mapeaveis
status_normalizado = F.upper(F.trim(F.regexp_replace(F.col("status"), "[-_]+", " ")))
status_filme = F.coalesce(status_map_expr[status_normalizado], F.lit("Não Informado"))

data_lancamento = F.coalesce(
    F.expr("try_to_date(release_date, 'yyyy-MM-dd')"),
    F.expr("try_to_date(release_date, 'MM-dd-yyyy')"),
    F.expr("try_to_date(release_date, 'dd/MM/yyyy')")
)

df_info = (
    df_info
    .withColumn("status_filme", status_filme)
    .withColumn("data_lancamento", data_lancamento)
    .withColumnRenamed("id", "id_filme")
    .withColumnRenamed("title", "titulo")
    .withColumnRenamed("original_title", "titulo_original")
    .withColumnRenamed("runtime", "duracao_minutos")
    .withColumnRenamed("original_language", "idioma_original")
    .withColumnRenamed("overview", "sinopse")
    .withColumnRenamed("tagline", "frase_divulgacao")
    .withColumn("ano_lancamento", F.year("data_lancamento"))
)

# Mantém apenas o registro mais recente.
janela_dedupe = Window.partitionBy("id_filme").orderBy(F.col("ingestion_datetime").desc())
df_info = (
    df_info
    .withColumn("_rn", F.row_number().over(janela_dedupe))
    .filter(F.col("_rn") == 1)
    .drop("_rn")
)

df_info_filmes = df_info.select(
    "id_filme", "titulo", "titulo_original", "data_lancamento", "ano_lancamento",
    "duracao_minutos", "idioma_original", "status_filme", "sinopse", "frase_divulgacao",
)

df_info_filmes.write.format("delta").mode("overwrite").saveAsTable("workspace.silver.tb_info_filmes")
display(df_info_filmes)

In [0]:
taxa_dolar = (
    spark.table("workspace.bronze.tb_cotacao_dolar")
    .orderBy(F.col("dataHoraCotacao").desc())
    .select("cotacaoCompra")
    .first()["cotacaoCompra"]
)
# usa a cotação de compra mais recente disponível na bronze
print("Taxa USD para BRL usada:", taxa_dolar)

def limpar_moeda(df, origem, destino):
    valor = F.upper(F.trim(F.col(origem)))
    valor = F.when(valor.isin("UNKNOWN", "NÃO INFORMADO", ""), F.lit(None)).otherwise(valor)

    # remove símbolos de moeda e separadores
    valor = F.regexp_replace(valor, "USD", "")
    valor = F.regexp_replace(valor, "[$,]", "")
    valor = F.trim(valor)

    # sufixo K/M (mil e milhão) multiplica pelos valores corretos
    multiplicador = (
        F.when(valor.endswith("K"), F.lit(1000))
        .when(valor.endswith("M"), F.lit(1000000))
        .otherwise(F.lit(1))
    )
    numero = F.regexp_replace(valor, "[KM]$", "")

    df = df.withColumn("_numero_moeda", numero)
    df = df.withColumn(destino, F.expr("try_cast(_numero_moeda AS DOUBLE)") * multiplicador)
    df = df.withColumn(destino, F.when(F.col(destino) > 0, F.col(destino)))

    return df.drop("_numero_moeda")

df_fin = spark.table("workspace.bronze.tb_movies_financials").withColumnRenamed("id", "id_filme")
df_fin = limpar_moeda(df_fin, "budget", "orcamento_usd")
df_fin = limpar_moeda(df_fin, "revenue", "receita_usd")

df_fin = (
    df_fin
    # calcula os valores equivalentes em reais
    .withColumn("orcamento_brl", F.col("orcamento_usd") * F.lit(taxa_dolar))
    .withColumn("receita_brl", F.col("receita_usd") * F.lit(taxa_dolar))
    # lucro só quando ambos os lados existem para evitar nulos.
    .withColumn(
        "lucro_usd",
        F.when(F.col("orcamento_usd").isNotNull() & F.col("receita_usd").isNotNull(),
                F.col("receita_usd") - F.col("orcamento_usd")),
    )
    .withColumn(
        "lucro_brl",
        F.when(F.col("orcamento_brl").isNotNull() & F.col("receita_brl").isNotNull(),
                F.col("receita_brl") - F.col("orcamento_brl")),
    )
    # margem percentual só quando orçamento > 0
    .withColumn(
        "margem_lucro_percentual",
        F.when(
            F.col("orcamento_usd").isNotNull() & (F.col("orcamento_usd") > 0) & F.col("lucro_usd").isNotNull(),
            (F.col("lucro_usd") / F.col("orcamento_usd")) * 100,
        ),
    )
    # orçamentos residuais muito baixos geram margem com mais de 10 dígitos e um cast comum quebraria a célula inteira
    .withColumn("margem_lucro_percentual", F.expr("try_cast(margem_lucro_percentual AS DECIMAL(18,2))"))
)

# tipo numérico decimal apropriado
colunas_decimais = ["orcamento_usd", "receita_usd", "orcamento_brl", "receita_brl", "lucro_usd", "lucro_brl"]
for c in colunas_decimais:
    df_fin = df_fin.withColumn(c, F.col(c).cast("decimal(18,2)"))

df_financeiro_filmes = df_fin.select(
    "id_filme", "orcamento_usd", "receita_usd", "orcamento_brl", "receita_brl",
    "lucro_usd", "lucro_brl", "margem_lucro_percentual",
)

df_financeiro_filmes.write.format("delta").mode("overwrite").saveAsTable("workspace.silver.tb_financeiro_filmes")
display(df_financeiro_filmes)

In [0]:
df_met = (
    spark.table("workspace.bronze.tb_movies_metrics")
    .withColumnRenamed("id", "id_filme")
    # Popularidade mistura separador decimal "," e "." na origem
    .withColumn("_popularidade_texto", F.regexp_replace(F.trim(F.col("popularity")), ",", "."))
    # column shift espalha texto/nomes vazados nas colunas de nota e voto, lixo vira NULL sem quebrar o pipeline
    .withColumn("popularidade", F.expr("try_cast(_popularidade_texto AS DOUBLE)"))
    .withColumn("nota_media_tmdb", F.expr("try_cast(vote_average AS DOUBLE)"))
    .withColumn("qtd_votos_tmdb", F.expr("try_cast(vote_count AS INT)"))
    .withColumn("nota_media_imdb", F.expr("try_cast(averageRating AS DOUBLE)"))
    .withColumn("qtd_votos_imdb", F.expr("try_cast(numVotes AS INT)"))
)

df_met = (
    df_met
    # Popularidade negativa é inválida vira NULL
    .withColumn("popularidade", F.when(F.col("popularidade") >= 0, F.col("popularidade")))
    # Nota fora de 0-10 (inclusive erro de escala) vira NULL, sem redividir
    .withColumn("nota_media_tmdb", F.when(F.col("nota_media_tmdb").between(0, 10), F.col("nota_media_tmdb")))
    .withColumn("nota_media_imdb", F.when(F.col("nota_media_imdb").between(0, 10), F.col("nota_media_imdb")))
    # Contagem de votos negativa é inválida e vira NULL
    .withColumn("qtd_votos_tmdb", F.when(F.col("qtd_votos_tmdb") >= 0, F.col("qtd_votos_tmdb")))
    .withColumn("qtd_votos_imdb", F.when(F.col("qtd_votos_imdb") >= 0, F.col("qtd_votos_imdb")))
)

df_metricas_engajamento = df_met.select(
    "id_filme", "popularidade", "nota_media_tmdb", "qtd_votos_tmdb", "nota_media_imdb", "qtd_votos_imdb",
)

df_metricas_engajamento.write.format("delta").mode("overwrite").saveAsTable("workspace.silver.tb_metricas_engajamento")
display(df_metricas_engajamento)

In [0]:
df_rev = (
    spark.table("workspace.bronze.tb_movies_reviews")
    .withColumnRenamed("id", "id_filme")
    .withColumnRenamed("nome", "nome_usuario")
    .withColumn("nota_usuario", F.expr("try_cast(nota AS DOUBLE)"))
    .withColumn(
        "comentario_usuario",
        F.when(F.trim(F.coalesce(F.col("comentario"), F.lit(""))) == "", F.lit("Sem comentário"))
        .otherwise(F.trim(F.col("comentario"))),
    )
)

# nota fora da escala permitida (0-10) é descartada como NULL
df_rev = df_rev.withColumn("nota_usuario", F.when(F.col("nota_usuario").between(0, 10), F.col("nota_usuario")))

df_avaliacoes_usuarios = (
    df_rev
    .select("id_filme", "nome_usuario", "nota_usuario", "comentario_usuario")
    # remove registros integralmente duplicados (mesmo filme+usuário+nota+comentário)
    .dropDuplicates(["id_filme", "nome_usuario", "nota_usuario", "comentario_usuario"])
)

df_avaliacoes_usuarios.write.format("delta").mode("overwrite").saveAsTable("workspace.silver.tb_avaliacoes_usuarios")
display(df_avaliacoes_usuarios)

In [0]:
GENEROS_VALIDOS = [
      "Action", "Adventure", "Animation", "Comedy", "Crime", "Documentary",
      "Drama", "Family", "Fantasy", "History", "Horror", "Music", "Mystery",
      "Romance", "Science Fiction", "TV Movie", "Thriller", "War", "Western",
  ]

# normaliza os separadores e divide os gêneros
df_generos = (
    spark.table("workspace.bronze.tb_credits_and_tags")
    .select(F.col("id").alias("id_filme"),F.col("genres"))
    .withColumn("genres",F.regexp_replace("genres", "[|;]", ","))
    .withColumn("genero",F.explode(F.split("genres", ",")))
)

# padroniza e restringe os gêneros conhecidos (TMDB). quando rodei teste apareceu muito lixo de column shift que é mais facil deixar claro os valores desejados
df_generos = (
    df_generos
    .withColumn("genero", F.initcap(F.trim("genero")))
    .filter(F.col("genero").isin(GENEROS_VALIDOS))
    .select("id_filme", "genero")
)

df_generos.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.silver.tb_generos")

display(df_generos)

In [0]:
mapeamento_tipo = {
    "cast": "Ator",
    "directors": "Diretor",
    "writers": "Roteirista",
    "production_companies": "Produtora"
}

def titulo_capitalizado(coluna):
    # initcap só reconhece espaço como separador de palavra sozinho, ele estraga nomes com hífen. Descobri depois que rodei e vi o Anya Taylor-joy sem capitalização
    partes = F.transform(F.split(coluna, "-"), lambda parte: F.initcap(parte))
    return F.array_join(partes, "-")

df_base = spark.table("workspace.bronze.tb_credits_and_tags")

# cria uma tabela para cada tipo de entidade
partes = []

for coluna, tipo in mapeamento_tipo.items():

    df_entidade = (
        df_base.select(F.col("id").alias("id_filme"),F.col(coluna).alias("bruto"))
        .withColumn("nome_entidade",F.explode(F.split("bruto", ",")))
        .withColumn("nome_entidade", titulo_capitalizado(F.trim(F.col("nome_entidade"))))
        .withColumn("tipo_entidade",F.lit(tipo))
        .filter(F.col("nome_entidade") != "")
        .filter(F.upper(F.col("nome_entidade")) != "N/A")
        .filter(F.length("nome_entidade") <= 60)
        .filter(~F.col("nome_entidade").rlike(r"^/.*\.(jpg|png)$"))
        .filter(~F.col("nome_entidade").rlike(r"^\d+(\.\d+)?$"))
        .filter(F.size(F.split("nome_entidade", " ")) <= 6)
        .select(
            "id_filme",
            "nome_entidade",
            "tipo_entidade"
        )
    )

    partes.append(df_entidade)

# une todos os tipos em uma única tabela e remove duplicatas
df_pessoas_empresas = partes[0]

for parte in partes[1:]:
    df_pessoas_empresas = df_pessoas_empresas.unionByName(parte)
df_pessoas_empresas = df_pessoas_empresas.dropDuplicates(
    ["id_filme", "nome_entidade", "tipo_entidade"]
)

df_pessoas_empresas.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.silver.tb_pessoas_empresas")

display(df_pessoas_empresas)

In [0]:
df_cot = (
    spark.table("workspace.bronze.tb_cotacao_dolar")
    .withColumn("data",F.expr("try_to_date(dataHoraCotacao)"))
    .withColumn("cotacao_dolar",F.col("cotacaoCompra").cast("double"))
    .select("data", "cotacao_dolar")
    .dropDuplicates(["data"])
)

# define o intervalo de datas disponível na Bronze
limites = df_cot.agg(
    F.min("data").alias("min_data"),
    F.max("data").alias("max_data")
).first()

# cria um calendário diário entre as duas datas
df_calendario = spark.sql(f"""
    SELECT explode(
        sequence(
            to_date('{limites["min_data"]}'),
            to_date('{limites["max_data"]}'),
            interval 1 day
        )
    ) AS data
""")

# preenche os dias sem cotação com o último valor disponível
janela_ffill = (
    Window
    .orderBy("data")
    .rowsBetween(
        Window.unboundedPreceding,
        Window.currentRow
    )
)

df_cotacao_dolar = (
    df_calendario
    .join(df_cot, "data", "left")
    .withColumn(
        "cotacao_dolar",
        F.last(
            "cotacao_dolar",
            ignorenulls=True
        ).over(janela_ffill)
    )
)

df_cotacao_dolar.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.silver.tb_cotacao_dolar")

display(df_cotacao_dolar)

#### Validação de qualidade

Estas células não fazem parte do pipeline, são checagens para confirmar que as regras de
negócio foram realmente aplicadas antes de seguir para a camada Gold. Mantive aqui pois esses testes foram fundamentais para encontrar o column shift do `tb_pessoas_empresas` e do `tb_generos`

In [0]:
spark.table("workspace.silver.tb_pessoas_empresas") \
      .filter(
          (F.length("nome_entidade") > 60) |
          F.col("nome_entidade").rlike(r"^/.*\.(jpg|png)$") |
          F.col("nome_entidade").rlike(r"^\d+(\.\d+)?$") |
          (F.size(F.split("nome_entidade", " ")) > 6)
      ) \
      .groupBy("tipo_entidade").count() \
      .show()

In [0]:
spark.table("workspace.silver.tb_generos") \
    .groupBy("genero").count() \
    .orderBy("count") \
    .show(30, truncate=False)

In [0]:
# tb_info_filmes
df = spark.table("workspace.silver.tb_info_filmes")
assert df.count() == df.select("id_filme").distinct().count(), "dedupe falhou em tb_info_filmes"
assert set(r[0] for r in df.select("status_filme").distinct().collect()) <= {
    "Lançado", "Pós-Produção", "Em Produção", "Planejado", "Rumores", "Cancelado", "Não Informado"
}, "status_filme fora do domínio traduzido"

# tb_financeiro_filmes
df = spark.table("workspace.silver.tb_financeiro_filmes")
assert df.filter("orcamento_usd <= 0").count() == 0, "orcamento_usd com valor zerado/negativo"
assert df.filter("receita_usd <= 0").count() == 0, "receita_usd com valor zerado/negativo"
assert df.filter(F.col("margem_lucro_percentual").isNull() & F.col("lucro_usd").isNotNull()).count() == 0, "margem estourou o decimal(18,2)"

# tb_metricas_engajamento
df = spark.table("workspace.silver.tb_metricas_engajamento")
assert df.filter("nota_media_tmdb NOT BETWEEN 0 AND 10").count() == 0, "nota_media_tmdb fora de 0-10"
assert df.filter("nota_media_imdb NOT BETWEEN 0 AND 10").count() == 0, "nota_media_imdb fora de 0-10"
assert df.filter("qtd_votos_tmdb < 0 OR qtd_votos_imdb < 0 OR popularidade < 0").count() == 0, "métrica negativa"

# tb_avaliacoes_usuarios
df = spark.table("workspace.silver.tb_avaliacoes_usuarios")
assert df.groupBy("id_filme", "nome_usuario", "nota_usuario", "comentario_usuario").count().filter("count > 1").count() == 0, "duplicata não removida"
assert df.filter(~F.col("nota_usuario").between(0, 10) & F.col("nota_usuario").isNotNull()).count() == 0, "nota_usuario fora de 0-10"
assert df.filter(F.trim(F.coalesce(F.col("comentario_usuario"), F.lit(""))) == "").count() == 0, "comentário vazio sem fallback"

# tb_generos
df = spark.table("workspace.silver.tb_generos")
assert df.filter((F.col("genero") == "") | F.col("genero").rlike("^[0-9]+$")).count() == 0, "resíduo em tb_generos"

# tb_pessoas_empresas
df = spark.table("workspace.silver.tb_pessoas_empresas")
assert df.filter((F.col("nome_entidade") == "") | (F.upper(F.col("nome_entidade")) == "N/A")).count() == 0, "resíduo em tb_pessoas_empresas"

# tb_cotacao_dolar
df = spark.table("workspace.silver.tb_cotacao_dolar")
assert df.filter(F.col("cotacao_dolar").isNull()).count() <= 1, "forward fill deixou dias sem cotação"

print("Todas as validações passaram.")